# Data Science Project - Phase 1: Data Collection
**The Context Gap**
Mohammad Qasim Khalaf Alsamarrae

In this notebook I collect the data for my project from three VizWiz datasets:
- VQA: questions asked by blind people about their photos, with 10 answers each
- Quality Issues: problems in each photo (blur, dark, framing...)
- Captions: 5 human descriptions for each photo

I use the validation split of all three.

## Download the data

In [ ]:
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip -O vqa.zip
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/image_quality/annotations.zip -O quality.zip
!wget -nc -q https://vizwiz.cs.colorado.edu/VizWiz_final/caption/annotations.zip -O captions.zip

!unzip -j -o -q vqa.zip -d vqa
!unzip -j -o -q quality.zip -d quality
!unzip -j -o -q captions.zip -d captions

In [ ]:
# Import Pandas and numpy
import numpy as np
import pandas as pd

## 1. VQA dataset (questions and answers)

In [ ]:
# the files are json not csv, so i use read_json
vqa = pd.read_json("vqa/val.json")
vqa.head()

In [ ]:
vqa.shape

In [ ]:
vqa.info()

In [ ]:
# one row: the question and its 10 answers
vqa.loc[0, "question"]

In [ ]:
vqa.loc[0, "answers"]

In [ ]:
# check empty values
vqa.isnull().sum()

In [ ]:
# check duplicated rows (only image and question, answers is a list)
vqa.duplicated(subset=["image", "question"]).sum()

In [ ]:
# 1 = answerable, 0 = unanswerable
vqa["answerable"].value_counts()

In [ ]:
# in percent
vqa["answerable"].value_counts(normalize=True) * 100

In [ ]:
vqa["answer_type"].value_counts()

## 2. Quality issues dataset

In [ ]:
quality = pd.read_json("quality/val.json")
quality.head()

In [ ]:
# flaws is a dictionary in one column, json_normalize splits it into columns
flaws = pd.json_normalize(quality["flaws"])
quality = pd.concat([quality.drop(columns="flaws"), flaws], axis=1)
quality.head()

Each number = how many of the 5 workers saw the problem.
BLR = blur, DRK = dark, BRT = too bright, FRM = bad framing, OBS = blocked, ROT = rotated, OTH = other, NON = no problem

In [ ]:
quality.shape

In [ ]:
quality.describe()

In [ ]:
quality.isnull().sum()

In [ ]:
# a problem counts if 3 or more of the 5 workers saw it
problems = ["BLR", "DRK", "BRT", "FRM", "OBS", "ROT", "OTH", "NON"]
percent = (quality[problems] >= 3).mean() * 100
percent.sort_values(ascending=False).round(1)

In [ ]:
percent.sort_values().plot(kind="barh", title="photo problems (%)");

## 3. Captions dataset

In [ ]:
# this file has two parts: images and annotations (the captions)
cap = pd.read_json("captions/val.json", typ="series")
cap_images = pd.DataFrame(cap["images"])
captions = pd.DataFrame(cap["annotations"])
cap_images.head()

In [ ]:
captions.head()

In [ ]:
print(cap_images.shape)
print(captions.shape)

In [ ]:
captions.isnull().sum()

## 4. Linking the three datasets
All three use the image file name, so I check how many questions are found in the other two datasets.

In [ ]:
# isin checks if each image is found in the other table
in_quality = vqa["image"].isin(quality["image"])
in_captions = vqa["image"].isin(cap_images["file_name"])

print("questions:", len(vqa))
print("found in quality:", in_quality.sum())
print("found in captions:", in_captions.sum())
print("found in both:", (in_quality & in_captions).sum())

In [ ]:
# photos with text in them (only the photos that have questions)
my_photos = cap_images[cap_images["file_name"].isin(vqa["image"])]
my_photos["text_detected"].mean() * 100

## 5. A look at the questions

In [ ]:
# number of words in each question
vqa["words"] = vqa["question"].str.split().str.len()
vqa["words"].describe()

In [ ]:
# the first two words of each question
vqa["start"] = vqa["question"].str.lower().str.split().str[:2].str.join(" ")
vqa["start"].value_counts().head(10)

## Save

In [ ]:
vqa.to_csv("vqa_raw.csv", index=False)
quality.to_csv("quality_raw.csv", index=False)
captions.to_csv("captions_raw.csv", index=False)
cap_images.to_csv("caption_images_raw.csv", index=False)

## Notes
- All 4,319 questions are found in both the quality and captions data, so nothing is missing.
- 32.1% of the questions could not be answered.
- The most common problem is bad framing (36.5%), then blur (29.9%). Only 29.4% of photos have no problem.
- 70.1% of the photos have text in them.
- Almost a third of the questions start with "what is".

Next step: cleaning the data and merging it into one table.